#### Copyright 2018 Google LLC.

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Systèmes de recommandation avec TensorFlow

Ce notebook, adapté d'un TP du [cours de Google sur les systèmes de recommandation](https://developers.google.com/machine-learning/recommendation/), apprend des embeddings d'utilisateurs et de films par factorisation de matrice, sur les notes de MovieLens.

# Introduction

Nous allons créer un système de recommandation de films à partir du jeu de données [MovieLens](https://movielens.org/), disponible [ici](http://grouplens.org/datasets/movielens/). Les données sont des notes de films (sur une échelle de 1 à 5).

## Plan
  1. Exploration des données MovieLens (10 minutes)
  1. Préliminaires (25 minutes)
  1. Entraînement d'un modèle de factorisation de matrice (15 minutes)
  1. Inspection des embeddings (15 minutes)
  1. Régularisation de la factorisation de matrice (15 minutes)
  1. Entraînement d'un modèle softmax (30 minutes)

## Mise en place

Commençons par importer les paquets nécessaires.

In [ ]:
# @title Imports (exécutez cette cellule)

import numpy as np
import pandas as pd
import collections
from mpl_toolkits.mplot3d import Axes3D
from IPython import display
from matplotlib import pyplot as plt
import sklearn
import sklearn.manifold
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior()
tf.logging.set_verbosity(tf.logging.ERROR)

# Ajout de quelques fonctions pratiques aux DataFrames Pandas.
pd.options.display.max_rows = 10
pd.options.display.float_format = "{:.3f}".format


def mask(df, key, function):
  """Returns a filtered dataframe, by applying function to key"""
  return df[function(df[key])]


def flatten_cols(df):
  df.columns = [" ".join(col).strip() for col in df.columns.values]
  return df


pd.DataFrame.mask = mask
pd.DataFrame.flatten_cols = flatten_cols

# Altair, déjà installé sur Colab, trace les graphiques interactifs.
import altair as alt

alt.data_transformers.enable("default", max_rows=None)

Nous téléchargeons ensuite les données MovieLens, et créons des DataFrames contenant les films, les utilisateurs et les notes.

In [ ]:
# @title Chargement des données MovieLens (exécutez cette cellule).

# Téléchargement des données MovieLens.
print("Téléchargement des données movielens...")
from urllib.request import urlretrieve
import zipfile

urlretrieve(
  "http://files.grouplens.org/datasets/movielens/ml-100k.zip", "movielens.zip"
)
zip_ref = zipfile.ZipFile("movielens.zip", "r")
zip_ref.extractall()
print("Terminé. Le jeu de données contient :")
print(zip_ref.read("ml-100k/u.info"))

# Chargement de chaque jeu de données (utilisateurs, films et notes).
users_cols = ["user_id", "age", "sex", "occupation", "zip_code"]
users = pd.read_csv("ml-100k/u.user", sep="|", names=users_cols, encoding="latin-1")

ratings_cols = ["user_id", "movie_id", "rating", "unix_timestamp"]
ratings = pd.read_csv(
  "ml-100k/u.data", sep="\t", names=ratings_cols, encoding="latin-1"
)

# Le fichier des films contient une caractéristique binaire par genre.
genre_cols = [
  "genre_unknown",
  "Action",
  "Adventure",
  "Animation",
  "Children",
  "Comedy",
  "Crime",
  "Documentary",
  "Drama",
  "Fantasy",
  "Film-Noir",
  "Horror",
  "Musical",
  "Mystery",
  "Romance",
  "Sci-Fi",
  "Thriller",
  "War",
  "Western",
]
movies_cols = [
  "movie_id",
  "title",
  "release_date",
  "video_release_date",
  "imdb_url",
] + genre_cols
movies = pd.read_csv("ml-100k/u.item", sep="|", names=movies_cols, encoding="latin-1")

# Les identifiants commençant à 1, on les décale pour qu'ils commencent à 0.
users["user_id"] = users["user_id"].apply(lambda x: str(x - 1))
movies["movie_id"] = movies["movie_id"].apply(lambda x: str(x - 1))
movies["year"] = movies["release_date"].apply(lambda x: str(x).split("-")[-1])
ratings["movie_id"] = ratings["movie_id"].apply(lambda x: str(x - 1))
ratings["user_id"] = ratings["user_id"].apply(lambda x: str(x - 1))
ratings["rating"] = ratings["rating"].apply(lambda x: float(x))

# Calcul du nombre de films auxquels chaque genre est attribué.
genre_occurences = movies[genre_cols].sum().to_dict()


# Certains films pouvant appartenir à plusieurs genres, on crée différentes
# colonnes 'genre' comme suit :
# - all_genres : tous les genres actifs du film.
# - genre : tiré au hasard parmi les genres actifs.
def mark_genres(movies, genres):
  def get_random_genre(gs):
    active = [genre for genre, g in zip(genres, gs) if g == 1]
    if len(active) == 0:
      return "Other"
    return np.random.choice(active)

  def get_all_genres(gs):
    active = [genre for genre, g in zip(genres, gs) if g == 1]
    if len(active) == 0:
      return "Other"
    return "-".join(active)

  movies["genre"] = [
    get_random_genre(gs) for gs in zip(*[movies[genre] for genre in genres])
  ]
  movies["all_genres"] = [
    get_all_genres(gs) for gs in zip(*[movies[genre] for genre in genres])
  ]


mark_genres(movies, genre_cols)

# Création d'un DataFrame fusionné contenant toutes les données movielens.
movielens = ratings.merge(movies, on="movie_id").merge(users, on="user_id")


# Utilitaire pour séparer les données en ensembles d'entraînement et de test.
def split_dataframe(df, holdout_fraction=0.1):
  """Splits a DataFrame into training and test sets.
  Args:
    df: a dataframe.
    holdout_fraction: fraction of dataframe rows to use in the test set.
  Returns:
    train: dataframe for training
    test: dataframe for testing
  """
  test = df.sample(frac=holdout_fraction, replace=False)
  train = df[~df.index.isin(test.index)]
  return train, test

# I. Exploration des données MovieLens
Avant de nous plonger dans la construction de modèles, inspectons notre jeu de données MovieLens. Il est généralement utile de comprendre les statistiques du jeu de données.

### Utilisateurs
Commençons par afficher quelques statistiques de base sur les caractéristiques numériques des utilisateurs.

In [ ]:
users.describe()

Nous pouvons aussi afficher quelques statistiques de base sur les caractéristiques catégorielles des utilisateurs

In [ ]:
users.describe(include=[object])

Nous pouvons aussi créer des histogrammes pour mieux comprendre la distribution des utilisateurs. Nous utilisons Altair pour créer un graphique interactif.

In [ ]:
# @title Code de visualisation Altair (exécutez cette cellule)
# Les fonctions suivantes servent à générer des graphiques Altair interactifs.
# Nous afficherons des histogrammes des données, découpées selon un attribut donné.

# Création des filtres qui serviront à découper les données.
occupation_filter = alt.selection_point(fields=["occupation"])
occupation_chart = (
  alt.Chart()
  .mark_bar()
  .encode(
    x="count()",
    y=alt.Y("occupation:N"),
    color=alt.condition(
      occupation_filter,
      alt.Color("occupation:N", scale=alt.Scale(scheme="category20")),
      alt.value("lightgray"),
    ),
  )
  .properties(width=300, height=300)
  .add_params(occupation_filter)
)


# Une fonction qui génère un histogramme des données filtrées.
def filtered_hist(field, label, filter):
  """Creates a layered chart of histograms.
  The first layer (light gray) contains the histogram of the full data, and the
  second contains the histogram of the filtered data.
  Args:
    field: the field for which to generate the histogram.
    label: String label of the histogram.
    filter: an alt.Selection object to be used to filter the data.
  """
  base = (
    alt.Chart()
    .mark_bar()
    .encode(
      x=alt.X(field, bin=alt.Bin(maxbins=10), title=label),
      y="count()",
    )
    .properties(
      width=300,
    )
  )
  return alt.layer(
    base.transform_filter(filter),
    base.encode(color=alt.value("lightgray"), opacity=alt.value(0.7)),
  ).resolve_scale(y="independent")

Regardons ensuite la distribution des notes par utilisateur. Cliquer sur une profession dans le graphique de droite filtre les données sur cette profession. L'histogramme correspondant est affiché en bleu, superposé à l'histogramme de l'ensemble des données (en gris clair). Vous pouvez utiliser MAJ+clic pour sélectionner plusieurs sous-ensembles.

Qu'observez-vous, et comment cela pourrait-il affecter les recommandations ?

In [ ]:
users_ratings = (
  ratings.groupby("user_id", as_index=False)
  .agg({"rating": ["count", "mean"]})
  .flatten_cols()
  .merge(users, on="user_id")
)

# Création d'un graphique pour le décompte et d'un autre pour la moyenne.
alt.hconcat(
  filtered_hist("rating count", "# notes / utilisateur", occupation_filter),
  filtered_hist("rating mean", "note moyenne par utilisateur", occupation_filter),
  occupation_chart,
  data=users_ratings,
)

### Films

Il est aussi utile de regarder les informations sur les films et leurs notes.

In [ ]:
movies_ratings = movies.merge(
  ratings.groupby("movie_id", as_index=False)
  .agg({"rating": ["count", "mean"]})
  .flatten_cols(),
  on="movie_id",
)

genre_filter = alt.selection_point(fields=["genre"])
genre_chart = (
  alt.Chart()
  .mark_bar()
  .encode(
    x="count()",
    y=alt.Y("genre"),
    color=alt.condition(genre_filter, alt.Color("genre:N"), alt.value("lightgray")),
  )
  .properties(height=300)
  .add_params(genre_filter)
)

In [ ]:
(
  movies_ratings[["title", "rating count", "rating mean"]]
  .sort_values("rating count", ascending=False)
  .head(10)
)

In [ ]:
(
  movies_ratings[["title", "rating count", "rating mean"]]
  .mask("rating count", lambda x: x > 20)
  .sort_values("rating mean", ascending=False)
  .head(10)
)

Enfin, le dernier graphique montre la distribution du nombre de notes et de la note moyenne.

In [ ]:
# Affichage du nombre de notes et de la note moyenne par film.
alt.hconcat(
  filtered_hist("rating count", "# notes / film", genre_filter),
  filtered_hist("rating mean", "note moyenne par film", genre_filter),
  genre_chart,
  data=movies_ratings,
)

# II. Préliminaires

Notre objectif est de factoriser la matrice des notes $M$ (les utilisateurs en lignes, les films en colonnes) en produit d'une matrice d'embeddings d'utilisateurs $U$ et d'une matrice d'embeddings de films $V$, telles que $M \approx UV^\top$ avec
$U = \begin{bmatrix} u_{1} \\ \hline \vdots \\ \hline u_{n} \end{bmatrix}$ et
$V = \begin{bmatrix} v_{1} \\ \hline \vdots \\ \hline v_{m} \end{bmatrix}$.

Ici
- $n$ est le nombre d'utilisateurs,
- $m$ est le nombre de films,
- $M_{ij}$ est la note du $j$-ième film par le $i$-ième utilisateur,
- chaque ligne $u_i$ de $U$ est un vecteur (embedding) de dimension $d$ représentant l'utilisateur $i$,
- chaque ligne $v_j$ de $V$ est un vecteur (embedding) de dimension $d$ représentant le film $j$,
- la prédiction du modèle pour la paire $(i, j)$ est le produit scalaire de leurs embeddings : $M_{ij} \approx u_i \cdot v_j$.

## Représentation creuse de la matrice des notes

La matrice des notes peut être très grande et, en général, la plupart de ses entrées ne sont pas observées, puisqu'un utilisateur donné ne note qu'un petit sous-ensemble des films. Pour une représentation efficace, nous utiliserons un [tf.SparseTensor](https://www.tensorflow.org/api_docs/python/tf/sparse/SparseTensor). Un `SparseTensor` représente la matrice à l'aide de trois tenseurs : `tf.SparseTensor(indices, values, dense_shape)` représente un tenseur où une valeur $M_{ij} = a$ est encodée en posant `indices[k] = [i, j]` et `values[k] = a`. Le dernier tenseur, `dense_shape`, précise la forme de la matrice complète sous-jacente.

#### Exemple jouet
Supposons que nous ayons $2$ utilisateurs et $4$ films. Notre DataFrame jouet contient trois notes :

user\_id | movie\_id | rating
--:|--:|--:
0 | 0 | 5.0
0 | 1 | 3.0
1 | 3 | 1.0

La matrice des notes correspondante est

$$
M =
\begin{bmatrix}
5.0 & 3.0 & 0 & 0 \\
0   &   0 & 0 & 1.0
\end{bmatrix}
$$

Et la représentation en SparseTensor est :
```python
SparseTensor(
  indices=[[0, 0], [0, 1], [1,3]],
  values=[5.0, 3.0, 1.0],
  dense_shape=[2, 4])
```

### Exercice 1 : construire une représentation tf.SparseTensor de la matrice des notes

Écrivez la fonction `build_rating_sparse_tensor`, qui transforme un DataFrame de notes (colonnes `user_id`, `movie_id` et `rating`) en `tf.SparseTensor` de forme `dense_shape` : par défaut, celle de la matrice des notes de MovieLens, `users.shape[0]` utilisateurs et `movies.shape[0]` films.

Vérifiez-la ensuite sur l'exemple jouet : construisez la matrice creuse de `toy_ratings` (définie dans la cellule suivante), de forme `[2, 4]`, convertissez-la en matrice dense avec `tf.sparse.to_dense`, et comparez le résultat à la matrice $M$ ci-dessus. Ce notebook utilise TensorFlow en mode graphe (l'API de TensorFlow 1) : un tenseur n'y prend de valeur que calculé par la méthode `run` d'une `tf.Session`.

**Documentation utile**

- [`tf.sparse.SparseTensor`](https://www.tensorflow.org/api_docs/python/tf/sparse/SparseTensor)
- [`tf.sparse.to_dense`](https://www.tensorflow.org/api_docs/python/tf/sparse/to_dense)
- [`tf.compat.v1.Session`](https://www.tensorflow.org/api_docs/python/tf/compat/v1/Session)

In [ ]:
toy_ratings = pd.DataFrame(
  {"user_id": ["0", "0", "1"], "movie_id": ["0", "1", "3"], "rating": [5.0, 3.0, 1.0]}
)


def build_rating_sparse_tensor(ratings_df, dense_shape=None):
  """
  Args:
    ratings_df: a pd.DataFrame with `user_id`, `movie_id` and `rating` columns.
    dense_shape: the shape [n, m] of the ratings matrix; by default, the number
      of users and the number of movies of MovieLens.
  Returns:
    A tf.SparseTensor representing the ratings matrix.
  """
  # Votre code ici
  ...


# Votre code ici : la vérification sur toy_ratings

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `ratings_df[["user_id", "movie_id"]].values` : un tableau de forme (nombre de notes, 2), la paire `[i, j]` de chaque note (les identifiants sont des chaînes, `"0"`, `"3"`, que TensorFlow convertit en entiers)
- `ratings_df["rating"].values` : les notes, dans le même ordre
- `tf.SparseTensor(indices=..., values=..., dense_shape=...)`
- `[users.shape[0], movies.shape[0]]` : la forme par défaut, quand `dense_shape` vaut `None`
- `with tf.Session() as sess:`, puis `sess.run(tenseur)` : la valeur d'un tenseur, un tableau NumPy

Sur l'exemple jouet, `tf.sparse.to_dense` doit redonner `[[5. 3. 0. 0.] [0. 0. 0. 1.]]`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def build_rating_sparse_tensor(ratings_df, dense_shape=None):
  if dense_shape is None:
    dense_shape = [users.shape[0], movies.shape[0]]
  indices = ratings_df[...].values
  values = ...
  return tf.SparseTensor(indices=indices, values=values, dense_shape=dense_shape)


toy_sparse = build_rating_sparse_tensor(toy_ratings, dense_shape=...)
with tf.Session() as sess:
  print(sess.run(tf.sparse.to_dense(...)))
```

#### Solution

In [ ]:
def build_rating_sparse_tensor(ratings_df, dense_shape=None):
  """
  Args:
    ratings_df: a pd.DataFrame with `user_id`, `movie_id` and `rating` columns.
    dense_shape: the shape [n, m] of the ratings matrix; by default, the number
      of users and the number of movies of MovieLens.
  Returns:
    A tf.SparseTensor representing the ratings matrix.
  """
  if dense_shape is None:
    dense_shape = [users.shape[0], movies.shape[0]]
  indices = ratings_df[["user_id", "movie_id"]].values
  values = ratings_df["rating"].values
  return tf.SparseTensor(indices=indices, values=values, dense_shape=dense_shape)


toy_sparse = build_rating_sparse_tensor(toy_ratings, dense_shape=[2, 4])
with tf.Session() as sess:
  print(sess.run(tf.sparse.to_dense(toy_sparse)))

## Calcul de l'erreur

Le modèle approxime la matrice des notes $M$ par un produit de rang faible $UV^\top$. Il nous faut un moyen de mesurer l'erreur d'approximation. Nous commencerons par utiliser l'erreur quadratique moyenne sur les entrées observées uniquement (nous y reviendrons plus tard). Elle est définie par

$$
\begin{align*}
\text{MSE}(M, UV^\top)
&= \frac{1}{|\Omega|}\sum_{(i, j) \in\Omega}{( M_{ij} - (UV^\top)_{ij})^2} \\
&= \frac{1}{|\Omega|}\sum_{(i, j) \in\Omega}{( M_{ij} - u_i \cdot v_j)^2}
\end{align*}
$$
où $\Omega$ est l'ensemble des notes observées, et $|\Omega|$ le cardinal de $\Omega$.

### Exercice 2 : erreur quadratique moyenne

Écrivez une fonction TensorFlow qui prend une matrice des notes creuse $M$ et les deux matrices d'embeddings $U, V$, et renvoie l'erreur quadratique moyenne $\text{MSE}(M, UV^\top)$, calculée sur les seules notes observées. Vérifiez-la sur l'exemple jouet, `toy_sparse`, avec les embeddings des conseils.

**Documentation utile**

- [`tf.linalg.matmul`](https://www.tensorflow.org/api_docs/python/tf/linalg/matmul)
- [`tf.gather_nd`](https://www.tensorflow.org/api_docs/python/tf/gather_nd)
- [`tf.compat.v1.losses.mean_squared_error`](https://www.tensorflow.org/api_docs/python/tf/compat/v1/losses/mean_squared_error)

In [ ]:
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  """
  Args:
    sparse_ratings: A SparseTensor rating matrix M, of dense_shape [n, m].
    user_embeddings: A dense Tensor U of shape [n, d] where d is the embedding
      dimension, such that row i, u_i, is the embedding of user i.
    movie_embeddings: A dense Tensor V of shape [m, d] where d is the embedding
      dimension, such that row j, v_j, is the embedding of movie j.
  Returns:
    A scalar Tensor representing the MSE between the true ratings and the
      model's predictions.
  """
  # Votre code ici
  ...


# Votre code ici : la vérification sur toy_sparse

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- dans cette section, on ne considère que les entrées observées pour calculer la perte
- `sparse_ratings.indices`, `sparse_ratings.values` : les paires $(i, j)$ observées et leurs notes
- `tf.matmul(user_embeddings, movie_embeddings, transpose_b=True)` : la matrice $UV^\top$ de toutes les prédictions
- `tf.gather_nd(matrice, indices)` : les entrées de `matrice` aux paires `indices`
- `tf.losses.mean_squared_error(notes, predictions)` : la moyenne des carrés des écarts

Sur l'exemple jouet, avec $U = \begin{bmatrix} 1 & 1 \\ 1 & 0 \end{bmatrix}$ et $V = \begin{bmatrix} 2 & 2 \\ 1 & 1 \\ 0 & 0 \\ 0 & 1 \end{bmatrix}$, les prédictions des trois notes observées (5, 3 et 1) sont 4, 2 et 0 : la MSE vaut 1.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  predictions = tf.gather_nd(tf.matmul(..., ..., transpose_b=True), ...)
  return tf.losses.mean_squared_error(..., predictions)


toy_U = tf.constant([[1.0, 1.0], [1.0, 0.0]])
toy_V = tf.constant([[2.0, 2.0], [1.0, 1.0], [0.0, 0.0], [0.0, 1.0]])
with tf.Session() as sess:
  print(sess.run(sparse_mean_square_error(toy_sparse, toy_U, toy_V)))
```

#### Solution

In [ ]:
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  """
  Args:
    sparse_ratings: A SparseTensor rating matrix M, of dense_shape [n, m].
    user_embeddings: A dense Tensor U of shape [n, d] where d is the embedding
      dimension, such that row i, u_i, is the embedding of user i.
    movie_embeddings: A dense Tensor V of shape [m, d] where d is the embedding
      dimension, such that row j, v_j, is the embedding of movie j.
  Returns:
    A scalar Tensor representing the MSE between the true ratings and the
      model's predictions.
  """
  predictions = tf.gather_nd(
    tf.matmul(user_embeddings, movie_embeddings, transpose_b=True),
    sparse_ratings.indices,
  )
  loss = tf.losses.mean_squared_error(sparse_ratings.values, predictions)
  return loss


toy_U = tf.constant([[1.0, 1.0], [1.0, 0.0]])
toy_V = tf.constant([[2.0, 2.0], [1.0, 1.0], [0.0, 0.0], [0.0, 1.0]])
with tf.Session() as sess:
  print(sess.run(sparse_mean_square_error(toy_sparse, toy_U, toy_V)))

Note : une approche consiste à calculer la matrice de prédiction complète $UV^\top$, puis à récupérer les entrées correspondant aux paires observées. Le coût mémoire de cette approche est en $O(nm)$. Pour le jeu de données MovieLens, cela ne pose pas de problème, car la matrice dense $n \times m$ est assez petite pour tenir en mémoire ($n = 943$, $m = 1682$).

Une autre approche (donnée dans la solution alternative ci-dessous) consiste à ne récupérer que les embeddings des paires observées, puis à calculer leurs produits scalaires. Le coût mémoire est en $O(|\Omega| d)$, où $d$ est la dimension des embeddings. Dans notre cas, $|\Omega| = 10^5$, et la dimension des embeddings est de l'ordre de $10$ : le coût mémoire des deux méthodes est donc comparable. Mais quand le nombre d'utilisateurs ou de films est bien plus grand, la première approche devient impraticable.

In [ ]:
# Solution alternative
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  """
  Args:
    sparse_ratings: A SparseTensor rating matrix M, of dense_shape [n, m].
    user_embeddings: A dense Tensor U of shape [n, d] where d is the embedding
      dimension, such that row i, u_i, is the embedding of user i.
    movie_embeddings: A dense Tensor V of shape [m, d] where d is the embedding
      dimension, such that row j, v_j, is the embedding of movie j.
  Returns:
    A scalar Tensor representing the MSE between the true ratings and the
      model's predictions.
  """
  predictions = tf.reduce_sum(
    tf.gather(user_embeddings, sparse_ratings.indices[:, 0])
    * tf.gather(movie_embeddings, sparse_ratings.indices[:, 1]),
    axis=1,
  )
  loss = tf.losses.mean_squared_error(sparse_ratings.values, predictions)
  return loss

### Exercice 3 (optionnel) : ajouter vos propres notes au jeu de données

Vous pouvez ajouter vos propres notes au jeu de données, pour obtenir ensuite des recommandations pour vous-même (section IV).

Saisissez vos notes dans le dictionnaire `my_ratings` de la cellule suivante : à chaque titre de film, écrit comme dans `movies["title"]`, une note de 1 à 5 ; les notes qu'il contient sont des exemples, à remplacer par les vôtres. Puis, dans la même cellule, ajoutez ces notes au DataFrame `ratings` avec `pd.concat`, sous l'identifiant d'utilisateur `"943"` (MovieLens en a 943, de `"0"` à `"942"`), et ajoutez une ligne à `users` pour ce nouvel utilisateur, pour que la matrice des notes ait une ligne de plus. Une deuxième exécution de la cellule doit remplacer vos notes, pas les ajouter une deuxième fois.

**Documentation utile**

- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

In [ ]:
# Vos notes : un titre de movies["title"], une note de 1 à 5.
my_ratings = {
  "Star Wars (1977)": 5,
  "Toy Story (1995)": 4,
  "Aladdin (1992)": 4,
  "Pulp Fiction (1994)": 2,
  "Fargo (1996)": 1,
}

# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `dict(zip(movies["title"], movies["movie_id"]))` : un dictionnaire qui donne l'identifiant (`movie_id`) de chaque titre
- `pd.DataFrame({"user_id": "943", "movie_id": [...], "rating": [...]})` : le DataFrame de vos notes ; une valeur seule, comme `"943"`, est répétée sur toutes les lignes
- `ratings[ratings["user_id"] != "943"]` : les notes, sans celles de l'utilisateur `"943"` (pour qu'une deuxième exécution remplace vos notes)
- `pd.concat([df1, df2], ignore_index=True)` : les lignes de `df1` puis celles de `df2`, renumérotées
- un titre mal orthographié lève une `KeyError` : `movies[movies["title"].str.contains("Star Wars")]` retrouve le titre exact

Avec les notes d'exemple, `ratings` passe de 100 000 à 100 005 lignes, et `users` de 943 à 944.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
title_to_id = dict(zip(movies["title"], movies["movie_id"]))
my_ratings_df = pd.DataFrame(
  {
    "user_id": "943",
    "movie_id": [title_to_id[title] for title in ...],
    "rating": [float(rating) for rating in ...],
  }
)
ratings = ratings[ratings["user_id"] != "943"]
ratings = pd.concat([..., ...], ignore_index=True)
if users.shape[0] == 943:
  users = pd.concat([users, pd.DataFrame({"user_id": ["943"]})], ignore_index=True)
```

#### Solution

In [ ]:
title_to_id = dict(zip(movies["title"], movies["movie_id"]))
my_ratings_df = pd.DataFrame(
  {
    "user_id": "943",
    "movie_id": [title_to_id[title] for title in my_ratings],
    "rating": [float(rating) for rating in my_ratings.values()],
  }
)
# Suppression des notes précédentes de l'utilisateur 943, puis ajout des nouvelles.
ratings = ratings[ratings["user_id"] != "943"]
ratings = pd.concat([ratings, my_ratings_df], ignore_index=True)
# Ajout du nouvel utilisateur au DataFrame des utilisateurs.
if users.shape[0] == 943:
  users = pd.concat([users, pd.DataFrame({"user_id": ["943"]})], ignore_index=True)
print("Vos %d notes ont été ajoutées ; vous avez très bon goût !" % len(my_ratings_df))
ratings[ratings["user_id"] == "943"].merge(movies[["movie_id", "title"]])

# III. Entraînement d'un modèle de factorisation de matrice

## Classe utilitaire CFModel (Collaborative Filtering Model)
Il s'agit d'une classe simple pour entraîner un modèle de factorisation de matrice par descente de gradient.

Le constructeur de la classe prend
- un dictionnaire des variables d'embeddings (des `tf.Variable`), par exemple `{"user_id": U, "movie_id": V}` ;
- une perte à optimiser (un `tf.Tensor`) ;
- une liste optionnelle de dictionnaires de métriques, chacun associant une chaîne (le nom de la métrique) à un tenseur. Ils sont évalués pendant l'entraînement, et chaque dictionnaire est tracé dans son propre graphique à la fin (par exemple l'erreur d'entraînement et l'erreur de test).

`model.train` renvoie la valeur des métriques à la dernière itération. Après l'entraînement, on peut accéder aux embeddings appris via le dictionnaire `model.embeddings`.

Exemple d'utilisation :
```
U = ...
V = ...
loss = ...
model = CFModel({"user_id": U, "movie_id": V}, loss, [{"train_error": loss}])
model.train(num_iterations=100, learning_rate=1.0)
user_embeddings = model.embeddings["user_id"]
movie_embeddings = model.embeddings["movie_id"]
```

In [ ]:
# @title Classe utilitaire CFModel (exécutez cette cellule)
class CFModel(object):
  """Simple class that represents a collaborative filtering model"""

  def __init__(self, embedding_vars, loss, metrics=None):
    """Initializes a CFModel.
    Args:
      embedding_vars: A dictionary of tf.Variables.
      loss: A float Tensor. The loss to optimize.
      metrics: optional list of dictionaries of Tensors. The metrics in each
        dictionary will be plotted in a separate figure during training.
    """
    self._embedding_vars = embedding_vars
    self._loss = loss
    self._metrics = metrics
    self._embeddings = {k: None for k in embedding_vars}
    self._session = None

  @property
  def embeddings(self):
    """The embeddings dictionary."""
    return self._embeddings

  def train(
    self,
    num_iterations=100,
    learning_rate=1.0,
    plot_results=True,
    optimizer=tf.train.GradientDescentOptimizer,
  ):
    """Trains the model.
    Args:
      num_iterations: number of iterations to run.
      learning_rate: optimizer learning rate.
      plot_results: whether to plot the results at the end of training.
      optimizer: the optimizer to use. Default to GradientDescentOptimizer.
    Returns:
      The metrics dictionary evaluated at the last iteration.
    """
    with self._loss.graph.as_default():
      opt = optimizer(learning_rate)
      train_op = opt.minimize(self._loss)
      local_init_op = tf.group(
        tf.variables_initializer(opt.variables()), tf.local_variables_initializer()
      )
      if self._session is None:
        self._session = tf.Session()
        with self._session.as_default():
          self._session.run(tf.global_variables_initializer())
          self._session.run(tf.tables_initializer())
          tf.train.start_queue_runners()

    with self._session.as_default():
      local_init_op.run()
      iterations = []
      metrics = self._metrics or ({},)
      metrics_vals = [collections.defaultdict(list) for _ in self._metrics]

      # Entraînement et ajout des résultats.
      for i in range(num_iterations + 1):
        _, results = self._session.run((train_op, metrics))
        if (i % 10 == 0) or i == num_iterations:
          print(
            "\r iteration %d: " % i
            + ", ".join(["%s=%f" % (k, v) for r in results for k, v in r.items()]),
            end="",
          )
          iterations.append(i)
          for metric_val, result in zip(metrics_vals, results):
            for k, v in result.items():
              metric_val[k].append(v)

      for k, v in self._embedding_vars.items():
        self._embeddings[k] = v.eval()

      if plot_results:
        # Affichage des métriques.
        num_subplots = len(metrics)
        fig = plt.figure()
        fig.set_size_inches(num_subplots * 10, 8)
        for i, metric_vals in enumerate(metrics_vals):
          ax = fig.add_subplot(1, num_subplots, i + 1)
          for k, v in metric_vals.items():
            ax.plot(iterations, v, label=k)
          ax.set_xlim([1, num_iterations])
          ax.legend()
      return results

## Construction du modèle

La fonction `build_model` de la cellule suivante construit un `CFModel` : elle sépare les notes en ensembles d'entraînement et de test, les représente en `SparseTensor` avec votre fonction `build_rating_sparse_tensor`, initialise les embeddings $U$ et $V$ selon une loi normale d'écart-type `init_stddev`, et définit les erreurs d'entraînement et de test avec votre fonction `sparse_mean_square_error`.

In [ ]:
# @title Construction du modèle (exécutez cette cellule)
def build_model(ratings, embedding_dim=3, init_stddev=1.0):
  """
  Args:
    ratings: a DataFrame of the ratings
    embedding_dim: the dimension of the embedding vectors.
    init_stddev: float, the standard deviation of the random initial embeddings.
  Returns:
    model: a CFModel.
  """
  # Séparation du DataFrame des notes en entraînement et test.
  train_ratings, test_ratings = split_dataframe(ratings)
  # Représentation en SparseTensor des jeux d'entraînement et de test.
  A_train = build_rating_sparse_tensor(train_ratings)
  A_test = build_rating_sparse_tensor(test_ratings)
  # Initialisation des embeddings selon une loi normale.
  U = tf.Variable(
    tf.random_normal([A_train.dense_shape[0], embedding_dim], stddev=init_stddev)
  )
  V = tf.Variable(
    tf.random_normal([A_train.dense_shape[1], embedding_dim], stddev=init_stddev)
  )
  train_loss = sparse_mean_square_error(A_train, U, V)
  test_loss = sparse_mean_square_error(A_test, U, V)
  metrics = {"train_error": train_loss, "test_error": test_loss}
  embeddings = {"user_id": U, "movie_id": V}
  return CFModel(embeddings, train_loss, [metrics])

Parfait, il est maintenant temps d'entraîner le modèle !

Exécutez la cellule suivante en essayant différents paramètres (dimension des embeddings, learning rate, itérations). Les erreurs d'entraînement et de test sont affichées à la fin de l'entraînement. Vous pouvez examiner ces valeurs pour valider les hyper-paramètres.

Note : en appelant à nouveau `model.train`, le modèle reprend l'entraînement à partir des valeurs actuelles des embeddings.

In [ ]:
# Construction du modèle CF et entraînement.
model = build_model(ratings, embedding_dim=30, init_stddev=0.5)
model.train(num_iterations=1000, learning_rate=10.0)

Le graphique trace les erreurs d'entraînement et de test au fil des itérations. La section IV examinera les embeddings appris plus en détail.

### Exercice 4 : dimension des embeddings et sur-apprentissage

Entraînez trois modèles avec `build_model`, en faisant varier `embedding_dim` : 3, 30 et 100, avec `init_stddev=0.5`, `learning_rate=10.0` et 300 itérations chacun (moins que les 1 000 de la cellule précédente, pour que l'exercice dure environ une minute). Pour chacun, relevez les erreurs d'entraînement et de test de la dernière itération.

Quand la dimension augmente, que deviennent l'erreur d'entraînement, l'erreur de test et leur écart ? À partir de quelle dimension le modèle sur-apprend-il ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `build_model(ratings, embedding_dim=..., init_stddev=0.5)` : un nouveau modèle ; donnez-lui un autre nom que `model`, que la suite du notebook utilise
- `.train(num_iterations=300, learning_rate=10.0, plot_results=False)` : l'entraînement, sans graphique ; il renvoie une liste d'un dictionnaire, `{"train_error": ..., "test_error": ...}`, les erreurs de la dernière itération
- `train` affiche sa progression sans retour à la ligne : un `print()` avant le vôtre

Les valeurs changent d'une exécution à l'autre (le découpage et l'initialisation sont aléatoires), mais pas leur ordre de grandeur : environ 0,85 en entraînement et 1 en test en dimension 3, 0,3 et 3,3 en dimension 100.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for embedding_dim in [3, 30, 100]:
  dim_model = build_model(ratings, embedding_dim=..., init_stddev=0.5)
  results = dim_model.train(num_iterations=300, learning_rate=10.0, plot_results=False)
  errors = results[0]
  print()
  print(embedding_dim, errors[...], errors[...])
```

#### Solution

In [ ]:
for embedding_dim in [3, 30, 100]:
  dim_model = build_model(ratings, embedding_dim=embedding_dim, init_stddev=0.5)
  results = dim_model.train(num_iterations=300, learning_rate=10.0, plot_results=False)
  errors = results[0]
  gap = errors["test_error"] - errors["train_error"]
  values = (embedding_dim, errors["train_error"], errors["test_error"], gap)
  print("\nDimension %d : entraînement %.2f, test %.2f, écart %.2f" % values)

- Quand la dimension augmente, l'erreur d'entraînement baisse et l'erreur de test monte : l'écart passe d'environ 0,1 en dimension 3 à plus de 0,5 en dimension 30, et à environ 3 en dimension 100.
- Le modèle sur-apprend dès la dimension 30 : un utilisateur a en moyenne une centaine de notes, dont 90 % à l'entraînement, et un embedding de 30 ou 100 coordonnées a assez de paramètres pour les apprendre par cœur, sans rien en tirer sur ses notes de test.
- Avec la dimension 3, le modèle généralise mieux, mais ses embeddings sont trop pauvres pour être intéressants à inspecter : la section V garde une grande dimension et ajoute une régularisation.

# IV. Inspection des embeddings

Dans cette section, nous examinons de plus près les embeddings appris :
- en calculant vos recommandations,
- en regardant les plus proches voisins de quelques films,
- en regardant les normes des embeddings de films,
- en visualisant les embeddings dans un espace projeté.

### Exercice 5 : écrire une fonction qui calcule les scores des candidats
Commençons par écrire une fonction qui, étant donnés un embedding de requête $u \in \mathbb R^d$ et des embeddings d'éléments $V \in \mathbb R^{m \times d}$ (une ligne $v_j$ par élément), calcule les scores des éléments.

Comme vu en cours, on peut utiliser différentes mesures de similarité, qui peuvent donner des résultats différents. Nous allons comparer les suivantes :
- produit scalaire : le score de l'élément $j$ est $u \cdot v_j$ ;
- cosinus : le score de l'élément $j$ est $\frac{u \cdot v_j}{\|u\|\|v_j\|}$.

**Documentation utile**

- [`numpy.dot`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html)
- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

In [ ]:
DOT = "dot"
COSINE = "cosine"


def compute_scores(query_embedding, item_embeddings, measure=DOT):
  """Computes the scores of the candidates given a query.
  Args:
    query_embedding: a vector of shape [k], representing the query embedding.
    item_embeddings: a matrix of shape [N, k], such that row i is the embedding
      of item i.
    measure: a string specifying the similarity measure to be used. Can be
      either DOT or COSINE.
  Returns:
    scores: a vector of shape [N], such that scores[i] is the score of item i.
  """
  # Votre code ici
  ...

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `u.dot(V.T)` : le vecteur des produits scalaires $u \cdot v_j$, un par élément
- `np.linalg.norm(V, axis=1, keepdims=True)` : la norme de chaque ligne de `V`, de forme `(m, 1)`, qui divise `V` ligne par ligne
- `np.linalg.norm(u)` : la norme de `u`
- avec le cosinus, normaliser `u` et chaque ligne de `V`, puis calculer leurs produits scalaires

Sur un petit exemple, avec $u = (1, 0)$ et deux éléments $v_1 = (2, 0)$ et $v_2 = (1, 1)$ : le produit scalaire donne 2 et 1, le cosinus 1 et 0,707 environ ($\cos 45° = 1/\sqrt 2$).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def compute_scores(query_embedding, item_embeddings, measure=DOT):
  u = query_embedding
  V = item_embeddings
  if measure == COSINE:
    V = V / np.linalg.norm(..., axis=1, keepdims=True)
    u = u / ...
  return u.dot(...)


print(compute_scores(np.array([1.0, 0.0]), np.array([[2.0, 0.0], [1.0, 1.0]]), DOT))
print(compute_scores(np.array([1.0, 0.0]), np.array([[2.0, 0.0], [1.0, 1.0]]), COSINE))
```

#### Solution

In [ ]:
DOT = "dot"
COSINE = "cosine"


def compute_scores(query_embedding, item_embeddings, measure=DOT):
  """Computes the scores of the candidates given a query.
  Args:
    query_embedding: a vector of shape [k], representing the query embedding.
    item_embeddings: a matrix of shape [N, k], such that row i is the embedding
      of item i.
    measure: a string specifying the similarity measure to be used. Can be
      either DOT or COSINE.
  Returns:
    scores: a vector of shape [N], such that scores[i] is the score of item i.
  """
  u = query_embedding
  V = item_embeddings
  if measure == COSINE:
    V = V / np.linalg.norm(V, axis=1, keepdims=True)
    u = u / np.linalg.norm(u)
  scores = u.dot(V.T)
  return scores


print(compute_scores(np.array([1.0, 0.0]), np.array([[2.0, 0.0], [1.0, 1.0]]), DOT))
print(compute_scores(np.array([1.0, 0.0]), np.array([[2.0, 0.0], [1.0, 1.0]]), COSINE))

### Recommandations et plus proches voisins

Munis de cette fonction, nous pouvons calculer des recommandations, où l'embedding de requête peut être celui d'un utilisateur ou celui d'un film.

In [ ]:
# @title Recommandations aux utilisateurs et plus proches voisins (exécutez cette cellule)
def user_recommendations(model, measure=DOT, exclude_rated=False, k=6):
  scores = compute_scores(
    model.embeddings["user_id"][943], model.embeddings["movie_id"], measure
  )
  score_key = measure + " score"
  df = pd.DataFrame(
    {
      score_key: list(scores),
      "movie_id": movies["movie_id"],
      "titles": movies["title"],
      "genres": movies["all_genres"],
    }
  )
  if exclude_rated:
    # suppression des films déjà notés
    rated_movies = ratings[ratings.user_id == "943"]["movie_id"].values
    df = df[df.movie_id.apply(lambda movie_id: movie_id not in rated_movies)]
  display.display(df.sort_values([score_key], ascending=False).head(k))


def movie_neighbors(model, title_substring, measure=DOT, k=6):
  # Recherche des identifiants de films correspondant à la sous-chaîne donnée.
  ids = movies[movies["title"].str.contains(title_substring)].index.values
  titles = movies.iloc[ids]["title"].values
  if len(titles) == 0:
    raise ValueError("Aucun film trouvé avec le titre %s" % title_substring)
  print("Plus proches voisins de : %s." % titles[0])
  if len(titles) > 1:
    print(
      "[Plusieurs films correspondent. Autres candidats : {}]".format(
        ", ".join(titles[1:])
      )
    )
  movie_id = ids[0]
  scores = compute_scores(
    model.embeddings["movie_id"][movie_id], model.embeddings["movie_id"], measure
  )
  score_key = measure + " score"
  df = pd.DataFrame(
    {score_key: list(scores), "titles": movies["title"], "genres": movies["all_genres"]}
  )
  display.display(df.sort_values([score_key], ascending=False).head(k))

### Vos recommandations

La cellule suivante calcule les recommandations de l'utilisateur 943, celui de l'exercice 3 : les vôtres, si vous y avez saisi vos notes.

In [ ]:
user_recommendations(model, measure=COSINE, k=5)

À quoi ressemblent les recommandations ?

### Plus proches voisins de films

Regardons les plus proches voisins de quelques films.

In [ ]:
movie_neighbors(model, "Aladdin", DOT)
movie_neighbors(model, "Aladdin", COSINE)

Il semble que la qualité des embeddings appris ne soit pas très bonne. Nous y remédierons dans la section V en ajoutant plusieurs techniques de régularisation. Mais d'abord, examinons davantage les embeddings.

## Norme des embeddings de films

On peut aussi observer que les recommandations avec le produit scalaire et avec le cosinus diffèrent : avec le produit scalaire, le modèle a tendance à recommander des films populaires. Cela s'explique par le fait que, dans les modèles de factorisation de matrice, la norme de l'embedding est souvent corrélée à la popularité (les films populaires ont une norme plus grande), ce qui rend plus probable la recommandation d'éléments populaires. Nous pouvons confirmer cette hypothèse en triant les films par norme de leur embedding, comme dans la cellule suivante.

In [ ]:
# @title Code de visualisation des embeddings (exécutez cette cellule)


def movie_embedding_norm(models):
  """Visualizes the norm and number of ratings of the movie embeddings.
  Args:
    model: A MFModel object.
  """
  if not isinstance(models, list):
    models = [models]
  df = pd.DataFrame(
    {
      "title": movies["title"],
      "genre": movies["genre"],
      "num_ratings": movies_ratings["rating count"],
    }
  )
  charts = []
  brush = alt.selection_interval()
  for i, model in enumerate(models):
    norm_key = "norm" + str(i)
    df[norm_key] = np.linalg.norm(model.embeddings["movie_id"], axis=1)
    nearest = alt.selection_point(
      encodings=["x", "y"],
      on="mouseover",
      nearest=True,
      empty=False,
      name="nearest%d" % i,
    )
    base = (
      alt.Chart()
      .mark_circle()
      .encode(
        x="num_ratings",
        y=norm_key,
        color=alt.condition(brush, alt.value("#4c78a8"), alt.value("lightgray")),
      )
      .add_params(nearest)
    )
    if i == 0:
      base = base.add_params(brush)
    text = (
      alt.Chart()
      .mark_text(align="center", dx=5, dy=-5)
      .encode(
        x="num_ratings", y=norm_key, text=alt.condition(nearest, "title", alt.value(""))
      )
    )
    charts.append(alt.layer(base, text))
  return alt.hconcat(*charts, data=df)


def visualize_movie_embeddings(data, x, y):
  nearest = alt.selection_point(
    encodings=["x", "y"], on="mouseover", nearest=True, empty=False
  )
  base = (
    alt.Chart()
    .mark_circle()
    .encode(
      x=x,
      y=y,
      color=alt.condition(genre_filter, "genre", alt.value("whitesmoke")),
    )
    .properties(width=600, height=600)
    .add_params(nearest)
  )
  text = (
    alt.Chart()
    .mark_text(align="left", dx=5, dy=-5)
    .encode(x=x, y=y, text=alt.condition(nearest, "title", alt.value("")))
  )
  return alt.hconcat(alt.layer(base, text), genre_chart, data=data)


def tsne_movie_embeddings(model):
  """Visualizes the movie embeddings, projected using t-SNE with Cosine measure.
  Args:
    model: A MFModel object.
  """
  tsne = sklearn.manifold.TSNE(
    n_components=2,
    perplexity=40,
    metric="cosine",
    early_exaggeration=10.0,
    init="pca",
    verbose=True,
    max_iter=400,
  )

  print("Exécution de t-SNE...")
  V_proj = tsne.fit_transform(model.embeddings["movie_id"])
  movies.loc[:, "x"] = V_proj[:, 0]
  movies.loc[:, "y"] = V_proj[:, 1]
  return visualize_movie_embeddings(movies, "x", "y")

In [ ]:
movie_embedding_norm(model)

Note : selon l'initialisation du modèle, vous pourrez observer que certains films de niche (avec peu de notes) ont une norme élevée, ce qui produit des recommandations aberrantes. Cela peut arriver si l'embedding de ce film est initialisé avec une norme élevée. Comme le film a peu de notes, il est rarement mis à jour et peut garder sa norme élevée. La régularisation atténuera ce problème.

### Exercice 6 : l'écart-type de l'initialisation

Construisez et entraînez un modèle `model_lowinit` comme `model`, mais avec un écart-type d'initialisation `init_stddev` dix fois plus petit, 0,05. Comparez ensuite ses plus proches voisins d'« Aladdin » (`movie_neighbors`) et les normes de ses embeddings (`movie_embedding_norm`) à ceux de `model`. Une grandeur utile : la norme attendue d'un vecteur de dimension $d$ dont les entrées suivent $\mathcal N(0, \sigma^2)$ vaut environ $\sigma \sqrt d$.

Comment cela affecte-t-il la distribution des normes des embeddings, et le classement des films de plus grande norme ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `build_model(ratings, embedding_dim=30, init_stddev=0.05)`, puis `train(num_iterations=1000, learning_rate=10.0)` : les paramètres de `model`, sauf l'écart-type
- `movie_neighbors(model_lowinit, "Aladdin", DOT)`, puis avec `COSINE`
- `movie_embedding_norm([model, model_lowinit])` : les deux graphiques côte à côte

En dimension 30, la norme initiale d'un embedding vaut environ $0{,}5 \sqrt{30} \approx 2{,}7$ avec `model`, et $0{,}05 \sqrt{30} \approx 0{,}27$ avec `model_lowinit`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
model_lowinit = build_model(ratings, embedding_dim=30, init_stddev=...)
model_lowinit.train(num_iterations=1000, learning_rate=10.0)
movie_neighbors(model_lowinit, "Aladdin", DOT)
movie_neighbors(model_lowinit, "Aladdin", COSINE)
movie_embedding_norm([model, ...])
```

#### Solution

In [ ]:
model_lowinit = build_model(ratings, embedding_dim=30, init_stddev=0.05)
model_lowinit.train(num_iterations=1000, learning_rate=10.0)
movie_neighbors(model_lowinit, "Aladdin", DOT)
movie_neighbors(model_lowinit, "Aladdin", COSINE)
movie_embedding_norm([model, model_lowinit])

## Visualisation des embeddings
Comme il est difficile de visualiser des embeddings dans un espace de grande dimension (quand la dimension des embeddings $k > 3$), une approche consiste à les projeter dans un espace de plus petite dimension. t-SNE (*T-distributed Stochastic Neighbor Embedding*) est un algorithme qui projette les embeddings en essayant de préserver leurs distances deux à deux. Il peut être utile pour la visualisation, mais il faut l'utiliser avec précaution. Pour en savoir plus sur l'utilisation de t-SNE, voir [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/).

In [ ]:
tsne_movie_embeddings(model_lowinit)

Vous pouvez mettre en évidence les embeddings d'un genre donné en cliquant sur le panneau des genres (MAJ+clic pour sélectionner plusieurs genres).

On observe que les embeddings ne semblent pas avoir de structure notable, et que les embeddings d'un genre donné sont répartis dans tout l'espace. Cela confirme la faible qualité des embeddings appris. L'une des principales raisons, que nous traiterons dans la section suivante, est que nous n'avons entraîné le modèle que sur les paires observées, et sans régularisation.

# V. Régularisation de la factorisation de matrice

Dans la section précédente, notre perte était définie comme l'erreur quadratique moyenne sur la partie observée de la matrice des notes. Cela pose problème : le modèle n'est jamais pénalisé sur les paires non observées, et peut donc rapprocher les embeddings de films qui n'ont rien à voir (que les mêmes utilisateurs n'ont jamais notés ensemble), qui obtiennent alors à tort des scores élevés. Ce phénomène est appelé *folding* (repliement).

Nous allons ajouter des termes de régularisation qui traitent ce problème. Nous utiliserons deux types de régularisation :
- La régularisation des paramètres du modèle. C'est un terme de régularisation $\ell_2$ classique sur les matrices d'embeddings : $r(U, V) =  \frac{1}{n} \sum_i \|u_i\|^2 + \frac{1}{m}\sum_j \|v_j\|^2$.
- Un a priori global qui pousse la prédiction de toute paire vers zéro, appelé terme de *gravité* : $g(U, V) = \frac{1}{nm} \sum_{i = 1}^n \sum_{j = 1}^m (u_i \cdot v_j)^2$.

La perte totale est alors donnée par
$$
\frac{1}{|\Omega|}\sum_{(i, j) \in \Omega} (M_{ij} - u_i \cdot v_j)^2 + \lambda _r r(U, V) + \lambda_g g(U, V)
$$
où $\lambda_r$ et $\lambda_g$ sont deux coefficients de régularisation (hyper-paramètres).

### Exercice 7 : construire un modèle de factorisation de matrice régularisé et l'entraîner
Complétez la fonction `build_regularized_model`, qui construit un modèle régularisé : les erreurs d'entraînement et de test sur les notes observées (`error_train`, `error_test`, avec votre fonction `sparse_mean_square_error`), le terme de gravité multiplié par $\lambda_g$ (`gravity_loss`) et le terme de régularisation $\ell_2$ multiplié par $\lambda_r$ (`regularization_loss`). Une fonction `gravity(U, V)` vous est fournie : elle calcule le terme de gravité à partir des deux matrices d'embeddings $U$ et $V$.

**Documentation utile**

- [`tf.math.reduce_sum`](https://www.tensorflow.org/api_docs/python/tf/math/reduce_sum)

In [ ]:
def gravity(U, V):
  """Creates a gravity loss given two embedding matrices."""
  return (
    1.0
    / (U.shape[0].value * V.shape[0].value)
    * tf.reduce_sum(
      tf.matmul(U, U, transpose_a=True) * tf.matmul(V, V, transpose_a=True)
    )
  )


def build_regularized_model(
  ratings, embedding_dim=3, regularization_coeff=0.1, gravity_coeff=1.0, init_stddev=0.1
):
  """
  Args:
    ratings: the DataFrame of movie ratings.
    embedding_dim: The dimension of the embedding space.
    regularization_coeff: The regularization coefficient lambda.
    gravity_coeff: The gravity regularization coefficient lambda_g.
  Returns:
    A CFModel object that uses a regularized loss.
  """
  # Séparation du DataFrame des notes en entraînement et test.
  train_ratings, test_ratings = split_dataframe(ratings)
  # Représentation en SparseTensor des jeux d'entraînement et de test.
  A_train = build_rating_sparse_tensor(train_ratings)
  A_test = build_rating_sparse_tensor(test_ratings)
  U = tf.Variable(
    tf.random_normal([A_train.dense_shape[0], embedding_dim], stddev=init_stddev)
  )
  V = tf.Variable(
    tf.random_normal([A_train.dense_shape[1], embedding_dim], stddev=init_stddev)
  )

  # Votre code ici : error_train, error_test, gravity_loss, regularization_loss
  total_loss = error_train + regularization_loss + gravity_loss
  losses = {
    "train_error_observed": error_train,
    "test_error_observed": error_test,
  }
  loss_components = {
    "observed_loss": error_train,
    "regularization_loss": regularization_loss,
    "gravity_loss": gravity_loss,
  }
  embeddings = {"user_id": U, "movie_id": V}

  return CFModel(embeddings, total_loss, [losses, loss_components])

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `sparse_mean_square_error(A_train, U, V)` : l'erreur d'entraînement ; avec `A_test`, celle de test
- `gravity_coeff * gravity(U, V)` : le terme de gravité, multiplié par son coefficient
- `tf.reduce_sum(U * U) / U.shape[0].value` : $\frac{1}{n} \sum_i \|u_i\|^2$, la somme des carrés des coefficients de $U$ divisée par son nombre de lignes ; de même pour $V$
- `regularization_coeff` : le coefficient $\lambda_r$ de la somme des deux

Un squelette des quatre lignes à écrire à la place de `# Votre code ici`, indentées comme le reste de la fonction, en remplaçant chaque `...` :

```python
error_train = sparse_mean_square_error(..., U, V)
error_test = ...
gravity_loss = gravity_coeff * ...
regularization_loss = regularization_coeff * (
  tf.reduce_sum(U * U) / U.shape[0].value + ...
)
```

#### Solution

In [ ]:
def gravity(U, V):
  """Creates a gravity loss given two embedding matrices."""
  return (
    1.0
    / (U.shape[0].value * V.shape[0].value)
    * tf.reduce_sum(
      tf.matmul(U, U, transpose_a=True) * tf.matmul(V, V, transpose_a=True)
    )
  )


def build_regularized_model(
  ratings, embedding_dim=3, regularization_coeff=0.1, gravity_coeff=1.0, init_stddev=0.1
):
  """
  Args:
    ratings: the DataFrame of movie ratings.
    embedding_dim: The dimension of the embedding space.
    regularization_coeff: The regularization coefficient lambda.
    gravity_coeff: The gravity regularization coefficient lambda_g.
  Returns:
    A CFModel object that uses a regularized loss.
  """
  # Séparation du DataFrame des notes en entraînement et test.
  train_ratings, test_ratings = split_dataframe(ratings)
  # Représentation en SparseTensor des jeux d'entraînement et de test.
  A_train = build_rating_sparse_tensor(train_ratings)
  A_test = build_rating_sparse_tensor(test_ratings)
  U = tf.Variable(
    tf.random_normal([A_train.dense_shape[0], embedding_dim], stddev=init_stddev)
  )
  V = tf.Variable(
    tf.random_normal([A_train.dense_shape[1], embedding_dim], stddev=init_stddev)
  )

  error_train = sparse_mean_square_error(A_train, U, V)
  error_test = sparse_mean_square_error(A_test, U, V)
  gravity_loss = gravity_coeff * gravity(U, V)
  regularization_loss = regularization_coeff * (
    tf.reduce_sum(U * U) / U.shape[0].value + tf.reduce_sum(V * V) / V.shape[0].value
  )
  total_loss = error_train + regularization_loss + gravity_loss
  losses = {
    "train_error_observed": error_train,
    "test_error_observed": error_test,
  }
  loss_components = {
    "observed_loss": error_train,
    "regularization_loss": regularization_loss,
    "gravity_loss": gravity_loss,
  }
  embeddings = {"user_id": U, "movie_id": V}

  return CFModel(embeddings, total_loss, [losses, loss_components])

### Entraînement du modèle régularisé

Il est maintenant temps d'entraîner le modèle régularisé ! Vous pouvez essayer différentes valeurs des coefficients de régularisation, et différentes dimensions d'embeddings.

In [ ]:
reg_model = build_regularized_model(
  ratings,
  regularization_coeff=0.1,
  gravity_coeff=1.0,
  embedding_dim=35,
  init_stddev=0.05,
)
reg_model.train(num_iterations=2000, learning_rate=20.0)

On observe que l'ajout des termes de régularisation augmente la MSE, sur l'entraînement comme sur le test. Cependant, comme nous allons le voir, la qualité des recommandations s'améliore. Cela met en évidence une tension entre l'ajustement aux données observées et la minimisation des termes de régularisation. L'ajustement aux données observées met souvent l'accent sur l'apprentissage de fortes similarités (entre éléments avec beaucoup d'interactions), mais une bonne représentation par embeddings demande aussi d'apprendre de faibles similarités (entre éléments avec peu ou pas d'interactions).

### Examen des résultats
Voyons si les résultats avec régularisation sont meilleurs.

In [ ]:
user_recommendations(reg_model, DOT, exclude_rated=True, k=10)

Avec un peu de chance, ces recommandations sont meilleures. Vous pouvez changer la mesure de similarité de COSINE à DOT et observer l'effet sur les recommandations.

Comme le modèle a tendance à recommander les éléments que vous avez bien notés, vous pouvez exclure les éléments que vous avez notés, avec `exclude_rated=True`.

Dans les cellules suivantes, nous affichons les plus proches voisins, les normes des embeddings et la projection t-SNE des embeddings de films.

In [ ]:
movie_neighbors(reg_model, "Aladdin", DOT)
movie_neighbors(reg_model, "Aladdin", COSINE)

Nous comparons ici les normes des embeddings de `model`, de `model_lowinit` et de `reg_model`. Sélectionner une zone du premier graphique met ses films en évidence sur tous les graphiques.

In [ ]:
movie_embedding_norm([model, model_lowinit, reg_model])

In [ ]:
# Visualisation des embeddings
tsne_movie_embeddings(reg_model)

On devrait observer que les embeddings sont bien plus structurés que sans régularisation. Essayez de sélectionner différents genres et observez comment ils ont tendance à former des groupes (par exemple Horror, Animation et Children).

### Conclusion
Ceci conclut cette section sur les modèles de factorisation de matrice. Notez que si la taille du problème est ici assez petite pour permettre un entraînement efficace par descente de gradient, de nombreux problèmes réels doivent être entraînés avec des algorithmes plus spécialisés, comme les moindres carrés alternés (ALS, *alternating least squares* : $V$ fixé, chaque $u_i$ est la solution d'un problème de moindres carrés, puis $U$ fixé, chaque $v_j$, et ainsi de suite), par exemple [l'ALS de Spark MLlib](https://spark.apache.org/docs/latest/ml-collaborative-filtering.html) ou la bibliothèque [implicit](https://benfred.github.io/implicit/).

# VI. Modèle softmax

Dans cette section, nous allons entraîner un modèle softmax simple qui prédit si un utilisateur donné a noté un film.

Le modèle prendra en entrée un vecteur de caractéristiques $x$ représentant la liste des films que l'utilisateur a notés. Nous partons du DataFrame des notes, que nous regroupons par user_id.

In [ ]:
rated_movies = (
  ratings[["user_id", "movie_id"]]
  .groupby("user_id", as_index=False)
  .aggregate(lambda x: list(x))
)
rated_movies.head()

Nous créons ensuite une fonction qui génère un batch d'exemples, où chaque exemple contient les caractéristiques suivantes :
- movie_id : un tenseur de chaînes, les identifiants des films que l'utilisateur a notés.
- genre : un tenseur de chaînes, les genres de ces films.
- year : un tenseur de chaînes, l'année de sortie.

In [ ]:
# @title Code de génération des batchs (exécutez cette cellule)
years_dict = {movie: year for movie, year in zip(movies["movie_id"], movies["year"])}
genres_dict = {
  movie: genres.split("-")
  for movie, genres in zip(movies["movie_id"], movies["all_genres"])
}


def make_batch(ratings, batch_size):
  """Creates a batch of examples.
  Args:
    ratings: A DataFrame of ratings such that examples["movie_id"] is a list of
      movies rated by a user.
    batch_size: The batch size.
  """

  def pad(x, fill):
    return pd.DataFrame.from_dict(x).fillna(fill).values

  movie = []
  year = []
  genre = []
  label = []
  for movie_ids in ratings["movie_id"].values:
    movie.append(movie_ids)
    genre.append([x for movie_id in movie_ids for x in genres_dict[movie_id]])
    year.append([years_dict[movie_id] for movie_id in movie_ids])
    label.append([int(movie_id) for movie_id in movie_ids])
  features = {
    "movie_id": pad(movie, ""),
    "year": pad(year, ""),
    "genre": pad(genre, ""),
    "label": pad(label, -1),
  }
  batch = (
    tf.data.Dataset.from_tensor_slices(features)
    .shuffle(1000)
    .repeat()
    .batch(batch_size)
    .make_one_shot_iterator()
    .get_next()
  )
  return batch


def select_random(x):
  """Selectes a random elements from each row of x."""

  def to_float(x):
    return tf.cast(x, tf.float32)

  def to_int(x):
    return tf.cast(x, tf.int64)

  batch_size = tf.shape(x)[0]
  rn = tf.range(batch_size)
  nnz = to_float(tf.count_nonzero(x >= 0, axis=1))
  rnd = tf.random_uniform([batch_size])
  ids = tf.stack([to_int(rn), to_int(nnz * rnd)], axis=1)
  return to_int(tf.gather_nd(x, ids))

### Fonction de perte
Le modèle softmax est un réseau de neurones qui transforme les caractéristiques d'entrée $x$ en un embedding d'utilisateur $\psi(x) \in \mathbb R^d$, où $d$ est la dimension des embeddings. Ce vecteur est ensuite multiplié par une matrice d'embeddings de films $V \in \mathbb R^{m \times d}$ (où $m$ est le nombre de films), et la sortie finale du modèle est le softmax du produit, une probabilité par film :
$$
\hat p(x) = \text{softmax}(\psi(x) V^\top).
$$
Étant donné une étiquette cible $y$, si l'on note $p = 1_y$ l'encodage one-hot de cette étiquette, la perte est l'entropie croisée entre $\hat p(x)$ et $p$ :
$$
-\sum_j p_j \log \hat p(x)_j = -\log \hat p(x)_y = -\left(\psi(x) \cdot v_y - \log \sum_{j = 1}^m e^{\psi(x) \cdot v_j}\right).
$$

### Exercice 8 : écrire une fonction de perte pour le modèle softmax

Écrivez une fonction qui prend les embeddings d'utilisateurs $\psi(x)$ d'un batch, les embeddings de films $V$ et les étiquettes cibles $y$ (pour chaque exemple, l'identifiant entier d'un film), et renvoie l'entropie croisée moyenne sur le batch. Écrivez-la à partir des `logits`, le produit $\psi(x) V^\top$ : leur log-softmax donne $\log \hat p(x)$, dont on garde, pour chaque exemple, la valeur de la classe cible, avant de prendre l'opposé de la moyenne. Vérifiez-la sur le petit exemple des conseils.

**Documentation utile**

- [`tf.nn.log_softmax`](https://www.tensorflow.org/api_docs/python/tf/nn/log_softmax)
- [`tf.one_hot`](https://www.tensorflow.org/api_docs/python/tf/one_hot)
- [`tf.math.reduce_mean`](https://www.tensorflow.org/api_docs/python/tf/math/reduce_mean)

In [ ]:
def softmax_loss(user_embeddings, movie_embeddings, labels):
  """Returns the cross-entropy loss of the softmax model.
  Args:
    user_embeddings: A tensor of shape [batch_size, embedding_dim].
    movie_embeddings: A tensor of shape [num_movies, embedding_dim].
    labels: An integer tensor of shape [batch_size], such that labels[i] is the
      target label (a movie id) for example i.
  Returns:
    The mean cross-entropy loss.
  """
  # Votre code ici
  ...


# Votre code ici : la vérification sur le petit exemple des conseils

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `tf.matmul(user_embeddings, movie_embeddings, transpose_b=True)` : les logits, de forme `[batch_size, num_movies]`
- `tf.nn.log_softmax(logits)` : $\log \hat p(x)$, ligne par ligne
- `tf.one_hot(labels, depth=tf.shape(logits)[1])` : l'encodage one-hot $1_y$ de chaque étiquette ; multiplié par le log-softmax, puis sommé sur l'axe 1 (`tf.reduce_sum(..., axis=1)`), il garde la valeur de la classe cible
- `tf.reduce_mean` : la moyenne sur le batch
- `tf.nn.sparse_softmax_cross_entropy_with_logits(logits=logits, labels=labels)` calcule la même perte, exemple par exemple : de quoi vérifier la vôtre

Sur un petit exemple de deux utilisateurs et trois films, avec $\psi(x) = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$, $V = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix}$ et les étiquettes $0$ et $2$ : les logits sont $\begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix}$, chaque exemple a une perte de $-(1 - \log(e + 1 + e)) = \log(1 + 2e) - 1$, et la perte moyenne vaut environ 0,862.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def softmax_loss(user_embeddings, movie_embeddings, labels):
  logits = tf.matmul(..., ..., transpose_b=True)
  log_probs = tf.nn.log_softmax(...)
  one_hot_labels = tf.one_hot(labels, depth=tf.shape(logits)[1])
  return -tf.reduce_mean(tf.reduce_sum(..., axis=1))


toy_user_embeddings = tf.constant([[1.0, 0.0], [0.0, 1.0]])
toy_movie_embeddings = tf.constant([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
toy_labels = tf.constant([0, 2])
with tf.Session() as sess:
  print(sess.run(softmax_loss(toy_user_embeddings, toy_movie_embeddings, toy_labels)))
```

#### Solution

In [ ]:
def softmax_loss(user_embeddings, movie_embeddings, labels):
  """Returns the cross-entropy loss of the softmax model.
  Args:
    user_embeddings: A tensor of shape [batch_size, embedding_dim].
    movie_embeddings: A tensor of shape [num_movies, embedding_dim].
    labels: An integer tensor of shape [batch_size], such that labels[i] is the
      target label (a movie id) for example i.
  Returns:
    The mean cross-entropy loss.
  """
  logits = tf.matmul(user_embeddings, movie_embeddings, transpose_b=True)
  log_probs = tf.nn.log_softmax(logits)
  one_hot_labels = tf.one_hot(labels, depth=tf.shape(logits)[1])
  loss = -tf.reduce_mean(tf.reduce_sum(one_hot_labels * log_probs, axis=1))
  return loss


toy_user_embeddings = tf.constant([[1.0, 0.0], [0.0, 1.0]])
toy_movie_embeddings = tf.constant([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
toy_labels = tf.constant([0, 2])
with tf.Session() as sess:
  print(sess.run(softmax_loss(toy_user_embeddings, toy_movie_embeddings, toy_labels)))
  # La même perte, calculée par TensorFlow en un appel
  toy_logits = tf.matmul(toy_user_embeddings, toy_movie_embeddings, transpose_b=True)
  print(
    sess.run(
      tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
          logits=toy_logits, labels=toy_labels
        )
      )
    )
  )

### Exercice 9 : construire un modèle softmax, l'entraîner et examiner ses embeddings

Nous sommes maintenant prêts à construire un CFModel softmax. L'architecture du modèle est définie dans la fonction `create_network` de la cellule suivante : les embeddings d'entrée (movie_id, genre et year) sont concaténés pour former la couche d'entrée, suivie de couches cachées dont les dimensions sont données par l'argument `hidden_dims`. La sortie de la dernière couche cachée, l'embedding d'utilisateur $\psi(x)$, est enfin multipliée par les embeddings de films pour obtenir les logits. Comme étiquette cible, nous utiliserons un movie_id tiré au hasard dans la liste des films que l'utilisateur a notés.

Le modèle, de l'entrée à la sortie :
1. trois caractéristiques de l'utilisateur : les films qu'il a notés, leurs genres et leurs années de sortie ;
2. pour chacune, un embedding en sac de mots, la moyenne des embeddings de ses valeurs ;
3. leur concaténation, puis les couches cachées, dont la sortie est $\psi(x)$ ;
4. les logits $\psi(x) V^\top$, un par film, puis leur softmax $\hat p(x)$, comparé à l'étiquette cible par la perte de l'exercice 8.

Complétez la fonction `build_softmax_model` : les pertes d'entraînement et de test, `train_loss` et `test_loss` (avec votre fonction `softmax_loss`), et la précision à 10 sur le test, `test_precision_at_10`, la proportion de l'étiquette cible parmi les 10 films de plus grands logits, en moyenne sur les exemples de test (au plus 0,1, puisqu'un exemple n'a qu'une cible). Les colonnes de caractéristiques que la fonction reçoit sont construites plus loin, par `make_embedding_col`.

**Documentation utile**

- [`tf.compat.v1.metrics.precision_at_k`](https://www.tensorflow.org/api_docs/python/tf/compat/v1/metrics/precision_at_k)

In [ ]:
def build_softmax_model(rated_movies, embedding_cols, hidden_dims):
  """Builds a Softmax model for MovieLens.
  Args:
    rated_movies: DataFrame of training examples.
    embedding_cols: A dictionary mapping feature names (string) to embedding
      column objects. This will be used in tf.feature_column.input_layer() to
      create the input layer.
    hidden_dims: int list of the dimensions of the hidden layers.
  Returns:
    A CFModel object.
  """

  def create_network(features):
    """Maps input features dictionary to user embeddings.
    Args:
      features: A dictionary of input string tensors.
    Returns:
      outputs: A tensor of shape [batch_size, embedding_dim].
    """
    # Création d'un embedding en sac de mots pour chaque caractéristique creuse.
    inputs = tf.feature_column.input_layer(features, embedding_cols)
    # Couches cachées.
    input_dim = inputs.shape[1].value
    for i, output_dim in enumerate(hidden_dims):
      w = (
        tf.get_variable(
          "hidden%d_w_" % i,
          shape=[input_dim, output_dim],
          initializer=tf.truncated_normal_initializer(stddev=1.0 / np.sqrt(output_dim)),
        )
        / 10.0
      )
      outputs = tf.matmul(inputs, w)
      input_dim = output_dim
      inputs = outputs
    return outputs

  train_rated_movies, test_rated_movies = split_dataframe(rated_movies)
  train_batch = make_batch(train_rated_movies, 200)
  test_batch = make_batch(test_rated_movies, 100)

  with tf.variable_scope("model", reuse=False):
    # Entraînement
    train_user_embeddings = create_network(train_batch)
    train_labels = select_random(train_batch["label"])
  with tf.variable_scope("model", reuse=True):
    # Test
    test_user_embeddings = create_network(test_batch)
    test_labels = select_random(test_batch["label"])
    movie_embeddings = tf.get_variable(
      "input_layer/movie_id_embedding/embedding_weights"
    )

  # Votre code ici : train_loss, test_loss, test_precision_at_10

  metrics = (
    {"train_loss": train_loss, "test_loss": test_loss},
    {"test_precision_at_10": test_precision_at_10},
  )
  embeddings = {"movie_id": movie_embeddings}
  return CFModel(embeddings, train_loss, metrics)

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `softmax_loss(train_user_embeddings, movie_embeddings, train_labels)` : la perte d'entraînement ; de même pour le test, avec `test_user_embeddings` et `test_labels`
- `tf.metrics.precision_at_k(labels=test_labels, predictions=..., k=10)` : `predictions` reçoit les logits du test, `tf.matmul(test_user_embeddings, movie_embeddings, transpose_b=True)`
- `precision_at_k` renvoie un couple `(precision, update_op)` : gardez `update_op`, qui met à jour la moyenne à chaque évaluation et la renvoie

Un squelette des trois lignes à écrire à la place de `# Votre code ici`, indentées comme le reste de la fonction, en remplaçant chaque `...` :

```python
train_loss = softmax_loss(...)
test_loss = softmax_loss(...)
_, test_precision_at_10 = tf.metrics.precision_at_k(
  labels=..., predictions=tf.matmul(..., ..., transpose_b=True), k=10
)
```

#### Solution

In [ ]:
def build_softmax_model(rated_movies, embedding_cols, hidden_dims):
  """Builds a Softmax model for MovieLens.
  Args:
    rated_movies: DataFrame of training examples.
    embedding_cols: A dictionary mapping feature names (string) to embedding
      column objects. This will be used in tf.feature_column.input_layer() to
      create the input layer.
    hidden_dims: int list of the dimensions of the hidden layers.
  Returns:
    A CFModel object.
  """

  def create_network(features):
    """Maps input features dictionary to user embeddings.
    Args:
      features: A dictionary of input string tensors.
    Returns:
      outputs: A tensor of shape [batch_size, embedding_dim].
    """
    # Création d'un embedding en sac de mots pour chaque caractéristique creuse.
    inputs = tf.feature_column.input_layer(features, embedding_cols)
    # Couches cachées.
    input_dim = inputs.shape[1].value
    for i, output_dim in enumerate(hidden_dims):
      w = (
        tf.get_variable(
          "hidden%d_w_" % i,
          shape=[input_dim, output_dim],
          initializer=tf.truncated_normal_initializer(stddev=1.0 / np.sqrt(output_dim)),
        )
        / 10.0
      )
      outputs = tf.matmul(inputs, w)
      input_dim = output_dim
      inputs = outputs
    return outputs

  train_rated_movies, test_rated_movies = split_dataframe(rated_movies)
  train_batch = make_batch(train_rated_movies, 200)
  test_batch = make_batch(test_rated_movies, 100)

  with tf.variable_scope("model", reuse=False):
    # Entraînement
    train_user_embeddings = create_network(train_batch)
    train_labels = select_random(train_batch["label"])
  with tf.variable_scope("model", reuse=True):
    # Test
    test_user_embeddings = create_network(test_batch)
    test_labels = select_random(test_batch["label"])
    movie_embeddings = tf.get_variable(
      "input_layer/movie_id_embedding/embedding_weights"
    )

  test_loss = softmax_loss(test_user_embeddings, movie_embeddings, test_labels)
  train_loss = softmax_loss(train_user_embeddings, movie_embeddings, train_labels)
  _, test_precision_at_10 = tf.metrics.precision_at_k(
    labels=test_labels,
    predictions=tf.matmul(test_user_embeddings, movie_embeddings, transpose_b=True),
    k=10,
  )

  metrics = (
    {"train_loss": train_loss, "test_loss": test_loss},
    {"test_precision_at_10": test_precision_at_10},
  )
  embeddings = {"movie_id": movie_embeddings}
  return CFModel(embeddings, train_loss, metrics)

### Entraînement du modèle softmax

Nous sommes maintenant prêts à entraîner le modèle softmax. Vous pouvez régler les hyper-paramètres suivants :
- le learning rate
- le nombre d'itérations. Note : vous pouvez exécuter à nouveau `softmax_model.train()` pour poursuivre l'entraînement du modèle à partir de son état actuel.
- les dimensions des embeddings d'entrée (le deuxième argument de chaque `make_embedding_col`, dans `embedding_cols`)
- le nombre de couches cachées et la taille de chaque couche (l'argument `hidden_dims`)

Note : comme nos caractéristiques d'entrée sont des chaînes (movie_id, genre et year), il faut les transformer en identifiants entiers. C'est le rôle de [`tf.feature_column.categorical_column_with_vocabulary_list`](https://www.tensorflow.org/api_docs/python/tf/feature_column/categorical_column_with_vocabulary_list), qui prend une liste de vocabulaire donnant toutes les valeurs que la caractéristique peut prendre. Chaque identifiant est ensuite associé à un vecteur d'embedding avec [`tf.feature_column.embedding_column`](https://www.tensorflow.org/api_docs/python/tf/feature_column/embedding_column).

In [ ]:
# Création des colonnes d'embeddings des caractéristiques
def make_embedding_col(key, embedding_dim):
  categorical_col = tf.feature_column.categorical_column_with_vocabulary_list(
    key=key, vocabulary_list=list(set(movies[key].values)), num_oov_buckets=0
  )
  return tf.feature_column.embedding_column(
    categorical_column=categorical_col,
    dimension=embedding_dim,
    # initialisation par défaut : loi normale tronquée d'écart-type 1/sqrt(dimension)
    combiner="mean",
  )


with tf.Graph().as_default():
  softmax_model = build_softmax_model(
    rated_movies,
    embedding_cols=[
      make_embedding_col("movie_id", 35),
      make_embedding_col("genre", 3),
      make_embedding_col("year", 2),
    ],
    hidden_dims=[35],
  )

softmax_model.train(
  learning_rate=8.0, num_iterations=3000, optimizer=tf.train.AdagradOptimizer
)

### Examen des embeddings

Nous pouvons examiner les embeddings de films comme pour les modèles précédents. Notez que dans ce cas, les embeddings de films servent à la fois d'embeddings d'entrée (pour la représentation en sac de mots de l'historique de l'utilisateur) et de poids du softmax.

In [ ]:
movie_neighbors(softmax_model, "Aladdin", DOT)
movie_neighbors(softmax_model, "Aladdin", COSINE)

In [ ]:
movie_embedding_norm([reg_model, softmax_model])

In [ ]:
tsne_movie_embeddings(softmax_model)

## Félicitations !

Vous avez terminé ce notebook Colab.

Si vous souhaitez explorer davantage ces modèles, nous vous encourageons à essayer différents hyper-paramètres et à observer leur effet sur la qualité du modèle et sur la structure de l'espace des embeddings. Quelques suggestions :
- Changez la dimension des embeddings.
- Dans le modèle softmax : changez le nombre de couches cachées et les caractéristiques d'entrée. Par exemple, vous pouvez essayer un modèle sans couche cachée, avec seulement les identifiants de films en entrée.
- Utilisez d'autres mesures de similarité : dans ce notebook Colab, nous avons utilisé le produit scalaire $d(u, v_j) = u \cdot v_j$ et le cosinus $d(u, v_j) = \frac{u \cdot v_j}{\|u\|\|v_j\|}$, et vu comment les normes des embeddings affectent les recommandations. Vous pouvez aussi essayer d'autres variantes qui appliquent une transformation à la norme, par exemple $d(u, v_j) = \frac{u \cdot v_j}{\|v_j\|^\alpha}$.